# Small Llama: Batch 2/5: 40% of Wikipedia

In [1]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import json
import random
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer, LlamaConfig, LlamaForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling
)
from datasets import load_dataset

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")

FRACTION_NAME = "40%"
FRACTION_VALUE = 0.4
SAFE_NAME = "40pct"

MODEL_DIR = Path(f"/content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_{SAFE_NAME}")
RESULTS_DIR = Path("/content/drive/MyDrive/Thesis/results/small_llama_batches")
LOCAL_BACKUP_DIR = Path("/content/local_results_backup")
TOKENIZED_DIR = Path(f"/content/drive/MyDrive/Thesis/data/tokenized_batches/{SAFE_NAME}")

MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESULTS_DIR / f"small_llama_{SAFE_NAME}_results.json"
LOCAL_RESULTS_PATH = LOCAL_BACKUP_DIR / f"small_llama_{SAFE_NAME}_results.json"

if RESULTS_PATH.exists():
    print(f"\u2713 Results already exist for {FRACTION_NAME} at {RESULTS_PATH}")
    print("Nothing to do \u2014 delete that file first if you want to rerun this batch.")


Mounted at /content/drive
GPU available: True
  Device: Tesla T4


## 1. Load Greek Wikipedia and select this batch's fraction

In [2]:
print("Loading Greek Wikipedia...")
ds = load_dataset("wikimedia/wikipedia", "20231101.el")
wiki = ds['train']
total_size = len(wiki)
print(f"Total articles: {total_size:,}")

subset_size = int(total_size * FRACTION_VALUE)
data_subset = wiki.select(range(subset_size))
print(f"Using {FRACTION_NAME}: {subset_size:,} articles")

Loading Greek Wikipedia...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.el/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  259MB            

20231101.el/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

20231101.el/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

20231101.el/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

20231101.el/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  187MB            

20231101.el/train-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/226834 [00:00<?, ? examples/s]

Total articles: 226,834
Using 40%: 90,733 articles


## 2. Tokenizer + tokenize this batch (cached, skips if already done)

In [3]:
print("Loading XLM-R tokenizer (handles accented Greek natively)...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding=False)

if TOKENIZED_DIR.exists():
    print(f"Loading cached tokenized data from {TOKENIZED_DIR}")
    from datasets import load_from_disk
    tokenized_data = load_from_disk(str(TOKENIZED_DIR))
else:
    print(f"Tokenizing {FRACTION_NAME}...")
    tokenized_data = data_subset.map(
        tokenize_function, batched=True,
        remove_columns=data_subset.column_names,
        desc=f"Tokenizing {FRACTION_NAME}"
    )
    tokenized_data.save_to_disk(str(TOKENIZED_DIR))

print(f"\u2713 {len(tokenized_data)} tokenized examples ready")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Loading XLM-R tokenizer (handles accented Greek natively)...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizing 40%...


Tokenizing 40%:   0%|          | 0/90733 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/90733 [00:00<?, ? examples/s]

✓ 90733 tokenized examples ready


## 3. ELBLiMP evaluation data + functions

In [4]:
def load_phenomenon_pairs(phenomenon_path):
    gram_file = phenomenon_path / "correct.txt"
    ungram_file = phenomenon_path / "incorrect.txt"
    with open(gram_file, 'r', encoding='utf-8') as f:
        gram_sentences = f.readlines()
    with open(ungram_file, 'r', encoding='utf-8') as f:
        ungram_sentences = f.readlines()
    return list(zip(gram_sentences, ungram_sentences))

def load_all_phenomena(data_dir):
    all_pairs = {}
    for folder in Path(data_dir).iterdir():
        if folder.is_dir():
            try:
                pairs = load_phenomenon_pairs(folder)
                all_pairs[folder.name] = pairs
                print(f"  \u2713 Loaded {folder.name}: {len(pairs)} pairs")
            except FileNotFoundError:
                print(f"  \u2717 Skipped {folder.name}: files not found")
    return all_pairs

all_data = load_all_phenomena("/content/drive/MyDrive/Thesis/data/phenomena")
print(f"\nTotal phenomena loaded: {len(all_data)}")

  ✓ Loaded noun_adjective_agreement: 100 pairs
  ✓ Loaded aspect: 100 pairs
  ✓ Loaded einai_agreement: 100 pairs
  ✓ Loaded subject_verb_agreement: 100 pairs
  ✓ Loaded negations: 100 pairs
  ✓ Loaded case_selection: 100 pairs

Total phenomena loaded: 6


In [5]:
class CausalLMScorer:
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def score_sentence(self, sentence):
        sentence = sentence.lower()
        input_ids = self.tokenizer.encode(sentence, return_tensors='pt')
        input_ids = input_ids.to(self.model.device)
        with torch.no_grad():
            outputs = self.model(input_ids, labels=input_ids)
            avg_log_prob = -outputs.loss.item()
        num_tokens = input_ids.shape[1]
        return {'avg_log_prob': avg_log_prob, 'avg_surprisal': -avg_log_prob, 'num_tokens': num_tokens}


def sample_size_stability(scorer, all_data, sample_sizes=[25, 50, 75, 100], n_repeats=5):
    results = {}
    for phenomenon, pairs in all_data.items():
        pair_correctness = []
        for gram, ungram in pairs:
            g = scorer.score_sentence(gram)
            u = scorer.score_sentence(ungram)
            pair_correctness.append(g['avg_log_prob'] > u['avg_log_prob'])

        phen_results = {}
        for size in sample_sizes:
            if size > len(pair_correctness):
                continue
            accs = []
            for rep in range(n_repeats):
                random.seed(rep)
                sample = random.sample(pair_correctness, size)
                accs.append(sum(sample) / size)
            phen_results[size] = {
                'mean_accuracy': sum(accs) / len(accs),
                'min_accuracy': min(accs),
                'max_accuracy': max(accs),
                'all_runs': accs
            }
        results[phenomenon] = phen_results
    return results


def full_evaluation(scorer, all_data):
    all_results = {}
    for phenomenon_name, pairs in all_data.items():
        correct_count = 0
        pair_results = []
        for gram, ungram in pairs:
            gram_result = scorer.score_sentence(gram)
            ungram_result = scorer.score_sentence(ungram)
            is_correct = gram_result['avg_log_prob'] > ungram_result['avg_log_prob']
            if is_correct:
                correct_count += 1
            pair_results.append({
                'grammatical': gram.strip(),
                'ungrammatical': ungram.strip(),
                'gram_avg_log_prob': gram_result['avg_log_prob'],
                'ungram_avg_log_prob': ungram_result['avg_log_prob'],
                'gram_surprisal': gram_result['avg_surprisal'],
                'ungram_surprisal': ungram_result['avg_surprisal'],
                'correct': is_correct
            })
        total_count = len(pairs)
        accuracy = correct_count / total_count
        all_results[phenomenon_name] = {
            'correct': correct_count,
            'total': total_count,
            'accuracy': accuracy,
            'avg_gram_log_prob': sum(p['gram_avg_log_prob'] for p in pair_results) / total_count,
            'avg_ungram_log_prob': sum(p['ungram_avg_log_prob'] for p in pair_results) / total_count,
            'avg_gram_surprisal': sum(p['gram_surprisal'] for p in pair_results) / total_count,
            'avg_ungram_surprisal': sum(p['ungram_surprisal'] for p in pair_results) / total_count,
            'pairs': pair_results
        }
        print(f"  {phenomenon_name}: {correct_count}/{total_count} = {accuracy:.2%}")
    return all_results

## 4. Train fresh model on 40%

In [6]:
print(f"{'='*70}\nTraining fresh model on {FRACTION_NAME} of Wikipedia\n{'='*70}")

# Fresh, randomly-initialized model \u2014 no pretraining, no continued
# training from another fraction. This is what makes the learning curve valid.
config = LlamaConfig(
    vocab_size=len(tokenizer),
    hidden_size=512,
    intermediate_size=1376,
    num_hidden_layers=8,
    num_attention_heads=8,
    max_position_embeddings=2048
)
model = LlamaForCausalLM(config)
print(f"Model has {model.num_parameters():,} parameters")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=10000,
    save_total_limit=1,
    logging_steps=500,
    learning_rate=5e-4,
    warmup_steps=500,
    fp16=True,
    logging_dir=str(MODEL_DIR / "logs")
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator,
)
trainer.train()

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"\u2713 Training on {FRACTION_NAME} complete! Saved to {MODEL_DIR}")

if torch.cuda.is_available():
    model = model.to('cuda')

Training fresh model on 40% of Wikipedia


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model has 281,307,648 parameters


Step,Training Loss
500,7.691479
1000,5.467979
1500,4.784184
2000,4.436294
2500,4.237567
3000,4.092343
3500,3.948693
4000,3.858309
4500,3.799526
5000,3.724834


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Training on 40% complete! Saved to /content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_40pct


## 5. Evaluate this checkpoint (full accuracy + sample-size stability)

In [7]:
scorer = CausalLMScorer(tokenizer, model)

print(f"\nRunning full evaluation for {FRACTION_NAME}...")
full_results = full_evaluation(scorer, all_data)

print(f"\nRunning sample-size stability for {FRACTION_NAME}...")
stability_results = sample_size_stability(scorer, all_data)

total_correct = sum(r['correct'] for r in full_results.values())
total_pairs = sum(r['total'] for r in full_results.values())

output = {
    'model': f"small_llama_{FRACTION_NAME}",
    'model_type': 'causal_lm',
    'training_data_fraction': FRACTION_NAME,
    'overall': {
        'total_pairs': total_pairs,
        'total_correct': total_correct,
        'accuracy': total_correct / total_pairs if total_pairs else 0
    },
    'per_phenomenon': full_results,
    'sample_size_stability': stability_results
}

print(f"\n{FRACTION_NAME} overall accuracy: {output['overall']['accuracy']:.2%}")


Running full evaluation for 40%...
  noun_adjective_agreement: 82/100 = 82.00%
  aspect: 65/100 = 65.00%
  einai_agreement: 77/100 = 77.00%
  subject_verb_agreement: 91/100 = 91.00%
  negations: 96/100 = 96.00%
  case_selection: 78/100 = 78.00%

Running sample-size stability for 40%...

40% overall accuracy: 81.50%


## 6. Save (Drive + local backup, both fsync'd)

In [8]:
# Save to Drive, forcing an actual flush (not just the FUSE buffer)
with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
    f.flush()
    os.fsync(f.fileno())
print(f"\u2713 Saved to Drive: {RESULTS_PATH}")

# Also save a local backup, independent of Drive sync
with open(LOCAL_RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
    f.flush()
    os.fsync(f.fileno())
print(f"\u2713 Saved local backup: {LOCAL_RESULTS_PATH}")

del model
gc.collect()
torch.cuda.empty_cache()
print("\n\u2713 Batch complete.")

✓ Saved to Drive: /content/drive/MyDrive/Thesis/results/small_llama_batches/small_llama_40pct_results.json
✓ Saved local backup: /content/local_results_backup/small_llama_40pct_results.json

✓ Batch complete.
